In [4]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
import random

# Step 1: Load data
data_path = "income.csv"
df = pd.read_csv(data_path)

# Step 2: Identify categorical, continuous, and label columns
categorical_columns = [
    "Workclass", "Education", "Marital Status", "Occupation",
    "Relationship", "Race", "Gender", "Native Country"
]
continuous_columns = [
    "Age", "Final Weight", "EducationNum", "Capital Gain",
    "capital loss", "Hours per Week"
]
label_column = "Income"

# Step 3: Data preprocessing
for col in categorical_columns:
    df[col] = df[col].str.strip()
    df[col] = df[col].replace('?', np.nan)
df = df.dropna(subset=categorical_columns + continuous_columns + [label_column])

df[label_column] = df[label_column].apply(lambda x: 1 if '>50K' in x else 0)

label_encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

scaler = StandardScaler()
df[continuous_columns] = scaler.fit_transform(df[continuous_columns])

cat_data = df[categorical_columns].values.astype(np.int64)
cont_data = df[continuous_columns].values.astype(np.float32)
labels = df[label_column].values.astype(np.int64)

cat_tensor = torch.tensor(cat_data)
cont_tensor = torch.tensor(cont_data)
label_tensor = torch.tensor(labels)

# Step 4: Split dataset 80/20 and use all rows
total_samples = len(df)
train_size = int(0.8 * total_samples)
test_size = total_samples - train_size

dataset = TensorDataset(cat_tensor, cont_tensor, label_tensor)
train_dataset, test_dataset = random_split(dataset, [train_size, test_size],
                                           generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=64, drop_last=False)

# Step 5: Define model
class TabularModel(nn.Module):
    def __init__(self, emb_dims, no_of_cont):
        super().__init__()
        self.emb_layers = nn.ModuleList([nn.Embedding(x, y) for x, y in emb_dims])
        self.emb_dropout = nn.Dropout(0.4)
        self.batch_norm_cont = nn.BatchNorm1d(no_of_cont)

        n_emb = sum([y for _, y in emb_dims])
        n_cont = no_of_cont

        self.layer1 = nn.Linear(n_emb + n_cont, 50)
        self.layer1_bn = nn.BatchNorm1d(50)
        self.dropout = nn.Dropout(0.4)
        self.output = nn.Linear(50, 2)

    def forward(self, x_cat, x_cont):
        embeddings = [emb_layer(x_cat[:, i]) for i, emb_layer in enumerate(self.emb_layers)]
        x_emb = torch.cat(embeddings, 1)
        x_emb = self.emb_dropout(x_emb)
        x_cont = self.batch_norm_cont(x_cont)
        x = torch.cat([x_emb, x_cont], 1)
        x = F.relu(self.layer1_bn(self.layer1(x)))
        x = self.dropout(x)
        x = self.output(x)
        return x

# Step 6: Prepare embeddings
emb_dims = []
for col in categorical_columns:
    num_unique = len(label_encoders[col].classes_)
    emb_dim = min(50, (num_unique + 1) // 2)
    emb_dims.append((num_unique, emb_dim))

model = TabularModel(emb_dims, len(continuous_columns))

# Step 7: Set seeds
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Step 8: Train
epochs = 300
model.train()

for epoch in range(epochs):
    epoch_loss = 0
    for x_cat_batch, x_cont_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(x_cat_batch, x_cont_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(train_loader):.4f}")

# Step 9: Evaluate
model.eval()
all_preds, all_labels = [], []
test_loss = 0
with torch.no_grad():
    for x_cat_batch, x_cont_batch, y_batch in test_loader:
        outputs = model(x_cat_batch, x_cont_batch)
        loss = criterion(outputs, y_batch)
        test_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        all_preds.append(preds)
        all_labels.append(y_batch)

avg_test_loss = test_loss / len(test_loader)
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)
accuracy = accuracy_score(all_labels.cpu(), all_preds.cpu())

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Prediction function (optional)
def predict_income(model, input_dict, label_encoders, scaler, categorical_columns, continuous_columns):
    model.eval()
    processed_cat, processed_cont = [], []
    for col in categorical_columns:
        val = input_dict.get(col, "")
        if val in label_encoders[col].classes_:
            processed_cat.append(label_encoders[col].transform([val])[0])
        else:
            processed_cat.append(0)
    for col in continuous_columns:
        processed_cont.append(input_dict.get(col, 0))
    processed_cont = scaler.transform([processed_cont])
    cat_tensor = torch.tensor([processed_cat], dtype=torch.int64)
    cont_tensor = torch.tensor(processed_cont, dtype=torch.float32)
    with torch.no_grad():
        output = model(cat_tensor, cont_tensor)
        pred = torch.softmax(output, dim=1)
        pred_label = torch.argmax(pred, dim=1).item()
        prob = pred[0][pred_label].item()
    return pred_label, prob


Epoch 50/300, Loss: 0.3390
Epoch 100/300, Loss: 0.3353
Epoch 150/300, Loss: 0.3364
Epoch 200/300, Loss: 0.3333
Epoch 250/300, Loss: 0.3323
Epoch 300/300, Loss: 0.3345
Test Loss: 0.3181
Test Accuracy: 0.8551


In [5]:

data_path = "income.csv"
df = pd.read_csv(data_path)


In [6]:
df.head()

,Age,Workclass,Final Weight,Education,EducationNum,Marital Status,Occupation,Relationship,Race,Gender,Capital Gain,capital loss,Hours per Week,Native Country,Income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [7]:
torch.save(model.state_dict(), "tabular_model_weights.pth")


In [8]:
# Recreate the model architecture
model = TabularModel(emb_dims, len(continuous_columns))
# Load weights
model.load_state_dict(torch.load("tabular_model_weights.pth"))
model.eval()


TabularModel(
  (emb_layers): ModuleList(
    (0): Embedding(7, 4)
    (1): Embedding(16, 8)
    (2): Embedding(7, 4)
    (3): Embedding(14, 7)
    (4): Embedding(6, 3)
    (5): Embedding(5, 3)
    (6): Embedding(2, 1)
    (7): Embedding(41, 21)
  )
  (emb_dropout): Dropout(p=0.4, inplace=False)
  (batch_norm_cont): BatchNorm1d(6, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer1): Linear(in_features=57, out_features=50, bias=True)
  (layer1_bn): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.4, inplace=False)
  (output): Linear(in_features=50, out_features=2, bias=True)
)

In [ ]:
name = "Dharshan D"
id_number = "212223230045"

print(name)
print(id_number)
